# Pulling measles vaccine data


Creates measles_vax_dates.csv, which is a csv of patientuid and date

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
# function that determines what kind of file we're dealing with and reads it into a pd DataFrame appropriately
def read_multi(path):
    # .csv
    if path.endswith('.csv'):
        data = pd.read_csv(path)
        return data
    
    # .csv.gz
    elif path.endswith('.csv.gz'):
        data = pd.read_csv(path, compression = 'gzip')
        return data
    
    elif path.endswith('.csv.zip'):
        data = pd.read_csv(path, compression='gzip', low_memory = False)
        return data
    
    # .pkl
    elif path.endswith('.pkl'):
        data = pd.read_pickle(path)
        return data
    
    else:
        print(path)

## Step 1: Subset relevant concept IDS

In [ ]:
file_name = "/share/pi/deho/AFC/BQ/concept_0524.csv.gz"
concepts = pd.read_csv(file_name)

In [ ]:
concepts = concepts[concepts['domain_id'].isin(['Procedure', 'Drug', 'Measurement', 'Condition'])] # cut to procedures and drugs only

In [ ]:
measles_concepts = ((concepts['concept_name'].str.contains("measles", case = False) &\
                   (concepts['concept_name'].str.contains("vacc", case = False) | \
                    concepts['concept_name'].str.contains("immuni", case = False))) |\
                    concepts['concept_name'].str.contains("mmr", case = False)) &\
                    (np.logical_not(concepts['concept_name'].str.contains("poison", case = False))) &\
                    (np.logical_not(concepts['concept_name'].str.contains("\[Presence\]", case = False))) &\
                    (np.logical_not(concepts['concept_name'].str.contains("\[Units/volume\]", case = False))) 
                # get all concepts containing (measles and vacc/imm) or (mmr) and not containing (poison)

In [ ]:
sum(measles_concepts)

In [ ]:
## subset out the domains of interest (drug, measurement, procedure, condition)
concepts[measles_concepts]['domain_id'].value_counts()

In [ ]:
measles_concepts_table = concepts[measles_concepts]

In [ ]:
measles_concepts_table.columns

In [ ]:
# create id and code tables
all_id = measles_concepts_table['concept_id']
all_code = measles_concepts_table['concept_code']
long_code = all_code[[len(x) >= 4 for x in all_code]]

In [ ]:
# save id and code tables
all_id.to_csv('../../tables/id_measles.csv', index = False)
all_code.to_csv('../../tables/code_measles.csv', index = False)

## Step 2: Go through each table, subset the rows of interest by person ID

### 2a: drug table

In [ ]:
path = "/share/pi/deho/AFC/BQ/"
path_list = os.listdir(path)
drug_paths = pd.Series(path_list)[pd.Series(path_list).str.startswith("drug")]

In [ ]:
# will eventually run this for every file in this list
drug = pd.DataFrame(columns = ['Unnamed: 0', 'person_id', 'drug_exposure_id',
       'drug_exposure_start_date', 'drug_concept_id', 'drug_source_value'])

for drug_name in drug_paths:
    drug_path = path + drug_name
    drug_data = pd.read_csv(drug_path)
    
    zero_drug = drug_data[drug_data['drug_concept_id'] == 0]
    drug_data = drug_data[drug_data['drug_concept_id'].isin(all_id)]
    
    zero_drug = zero_drug[[('measles' in str(x).lower()) | ('mmr' in str(x).lower())  for x in zero_drug['drug_source_value']]]
    
    drug = pd.concat([drug, drug_data, zero_drug], ignore_index=True)
    
    print(drug_name)

In [ ]:
len(drug)

In [ ]:
len(drug['person_id'].unique())

In [ ]:
sum(drug['drug_concept_id'] == 0)

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_drug.csv', drug)

### 2b: procedure table -- is not picking up anything for measles, for some reason. Have commented out.

In [ ]:
path = "/share/pi/deho/AFC/BQ/"
path_list = os.listdir(path)
proc_paths = pd.Series(path_list)[pd.Series(path_list).str.startswith("proc")]

In [ ]:
# will eventually run this for every file in this list
proc = pd.DataFrame(columns = ['procedure_occurrence_id', 'person_id', 'procedure_concept_id',
       'procedure_date', 'procedure_type_concept_id', 'visit_occurrence_id', 'procedure_source_concept_id',
                              'procedure_source_value'])

for proc_name in proc_paths:
    proc_path = path + proc_name
    proc_data = pd.read_csv(proc_path, low_memory = False)
    
    # concept id is exactly on the list
    id_match = proc_data[proc_data['procedure_source_concept_id'].astype(int).isin(all_id)]

    # concept id is 0, but code appears to match
    zero_proc = proc_data[proc_data['procedure_source_concept_id'] == 0]
    # code contains measles or mmr
    code_mmr = [('measles' in str(x).lower())|('mmr' in str(x).lower()) for x in zero_proc['procedure_source_value']]
    # must start with a code, then '|', or must be a code
    start_zero_proc = [any(x.startswith(code+'|') for code in long_code) for x in zero_proc['procedure_source_value']]
    # or must exactly be the code
    is_zero_proc = zero_proc['procedure_source_value'].isin(all_code)
    # subset to those being or containing code
    zero_proc = zero_proc[start_zero_proc or is_zero_proc or code_mmr]
    
    proc = pd.concat([proc, id_match, zero_proc], ignore_index=True)
    
    print(proc_name)

In [ ]:
proc

In [ ]:
len(proc)

In [ ]:
len(proc['person_id'].unique())

In [ ]:
sum(proc['procedure_source_concept_id'] == 0)

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_proc.csv', proc)

In [ ]:
## Prepare lists to store results
#proc_list = []

## Combine the code lists into single regex strings to minimize multiple `str.contains` calls
#codes = '|'.join(pd.concat([drug_code, proc_code]))

#for proc_name in proc_paths:
#    proc_path = path + proc_name
#    proc_data = pd.read_csv(proc_path)
#    
#    # Use `str.contains` once with a combined regex pattern for booster
#    indic = proc_data['procedure_source_value'].str.contains(codes, regex=False, na=False)
#    proc_data = proc_data[
#        proc_data['procedure_source_concept_id'].isin(proc_id) |
#        proc_data['procedure_source_concept_id'].isin(drug_id) |
#        indic
#    ]
#    print("done")
#    
#    # Append results to the lists
#    proc_list.append(proc_data)
#
## Concatenate the results in a single operation
#proc = pd.concat(proc_list, ignore_index=True)

## 2c: Measurement

In [ ]:
path = "/share/pi/deho/AFC/BQ/"
path_list = os.listdir(path)
meas_paths = pd.Series(path_list)[pd.Series(path_list).str.startswith("measure") & pd.Series(path_list).str.endswith("measles.csv.gz")]

In [ ]:
meas = pd.DataFrame(columns = ['Unnamed: 0', 'person_id', 'visit_occurrence_id',
       'measurement_concept_id', 'measurement_date', 'operator_concept_id',
       'value_as_number', 'unit_concept_id', 'range_low', 'range_high',
       'measurement_source_value'])

for meas_name in meas_paths: #meas_name = meas_paths[64]
    meas_path = path + meas_name
    meas_data = pd.read_csv(meas_path, low_memory = False)
    meas = pd.concat([meas, meas_data], ignore_index=True)

In [ ]:
meas

In [ ]:
len(meas)

In [ ]:
len(meas['person_id'].unique())

In [ ]:
sum(meas['measurement_concept_id'] == 0)

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_measurement.csv', meas)

## 2d: Condition

In [ ]:
path = "/share/pi/deho/AFC/BQ/"
path_list = os.listdir(path)
cond_paths = pd.Series(path_list)[pd.Series(path_list).str.startswith("condition_occurrence") &\
                                  (pd.Series(path_list).str.contains(".pkl")| pd.Series(path_list).str.contains(".csv"))]

In [ ]:
# will eventually run this for every file in this list
cond = pd.DataFrame(columns = ['person_id', 'condition_concept_id', 'condition_start_date',
       'condition_type_concept_id', 'visit_occurrence_id',
       'condition_source_concept_id', 'condition_source_value'])

for cond_name in cond_paths:
    cond_path = path + cond_name
    cond_data = read_multi(cond_path)
    
    zero_cond = cond_data[cond_data['condition_concept_id'] == 0]
    cond_data = cond_data[cond_data['condition_concept_id'].isin(all_id)]
    
    zero_cond_word = [('measles' in str(x).lower()) | ('mmr' in str(x).lower()) for x in zero_cond['condition_source_value']]
    zero_cond_code = [any(str(x).startswith(code+'|') for code in long_code) for x in zero_cond['condition_source_value']]
    zero_cond = zero_cond[zero_cond_word or zero_cond_code]
    
    cond = pd.concat([cond, cond_data, zero_cond], ignore_index=True)
    
    print(cond_name)

In [ ]:
cond

In [ ]:
len(cond)

In [ ]:
len(cond['person_id'].unique())

In [ ]:
sum(cond['condition_concept_id'] == 0)

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_condition.csv', cond)

## 3: Reload the created datasets and restrict to actual vaccine events

In [ ]:
# Drug
drug = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_drug.csv.zip')
# look at ids and codes
drug_nonzero = drug[drug['drug_concept_id'] != 0]
drug_zero = drug[drug['drug_concept_id'] == 0]

# look up by hand to make sure they're all valid measles vaccine IDs
# print(drug_nonzero['drug_concept_id'].value_counts())

# list of things we don't want in the drug source value
drug_omit = ['mmrn', 'mmref', 'hemmroid', 'hemmrhoid', 'immrel', 'post mmr', 'blood work']
drug_zero_filtered = drug_zero[[not any(omit in x.lower() for omit in drug_omit) for x in drug_zero['drug_source_value']]]

drug = pd.concat([drug_nonzero, drug_zero_filtered])
# restrict to only person id and date columns
drug = drug[['person_id', 'drug_exposure_start_date']]
drug = drug.loc[drug.notna().all(axis='columns')]
drug = drug.rename(columns={"drug_exposure_start_date": "date"}, errors="raise")

drug

In [ ]:
# Procedure
proc = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_proc.csv.zip')
# look at ids and codes

proc_nonzero = proc[proc['procedure_source_concept_id'] != 0]
proc_zero = proc[proc['procedure_source_concept_id'] == 0]

# look up by hand to make sure they're all valid measles vaccine IDs
# print(proc_nonzero['procedure_source_value'].value_counts())
# in this case, they're all related to serum globulin which is used for treatment, not vaccination

# list of things we don't want in the procedure source value
proc_omit = ['v06.4', 'v04.2', 'nasal']
proc_zero_filtered = proc_zero[[not any(omit in x.lower() for omit in proc_omit) for x in proc_zero['procedure_source_value']]]

proc = proc_zero_filtered

# restrict to only person id and date columns
proc = proc[['person_id', 'procedure_date']]
proc = proc.loc[proc.notna().all(axis='columns')]
proc = proc.rename(columns={"procedure_date": "date"}, errors="raise")

In [ ]:
# Measurement 
meas = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_measurement.csv.zip')
# all restrictions on codes etc are handled in the regex

# restrict to only person id and date columns
meas = meas[['person_id', 'measurement_date']]
meas = meas.loc[meas.notna().all(axis='columns')]
meas = meas.rename(columns={"measurement_date": "date"}, errors="raise")

In [ ]:
# Condition
cond = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_condition.csv.zip')

cond_zero = cond[cond['condition_source_concept_id']==0]
cond_nonzero = cond[cond['condition_source_concept_id']!=0]

cond_omit = ['v73.2', 'v06.4', 'need', 'screening', 'v22.2', 'declines', 'hemmroid', 'hemmrhoid', 'unknown', 'come back', 'follow up', 'allergy', 'pending']
cond_zero_filtered = cond_zero[[not any(omit in x.lower() for omit in cond_omit) for x in cond_zero['condition_source_value']]]

cond = cond_zero_filtered

# restrict to only person id and date columns
cond = cond[['person_id', 'condition_start_date']]
cond = cond.loc[cond.notna().all(axis='columns')]
cond = cond.rename(columns={"condition_start_date": "date"}, errors="raise")

## 4: Merge the datasets and restrict to unique rows

In [ ]:
vaccines = pd.concat([drug, proc, meas, cond]).drop_duplicates()

In [ ]:
vaccines = vaccines.sort_values(by=['person_id', 'date'])
# subset to first vaccine
vaccines_first = vaccines.groupby('person_id').first().reset_index()
vaccines_first

In [ ]:
# add patientuid
person = pd.read_csv('/share/pi/deho-pi/AFC/BQ/person_0524.csv.gz')
person = person[['person_id', 'person_source_value']]
vaccines_merged = pd.merge(vaccines_first, person)

In [ ]:
vaccines_merged = vaccines_merged.rename(columns={"person_source_value": "patientuid"}, errors="raise")
vaccines_merged = vaccines_merged[['patientuid', 'date']]

In [ ]:
len(vaccines_merged)

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax.csv', vaccines_merged)

This script used to export '/share/pi/deho-pi/AFC/measles_vax_dates.csv.zip', which was similar data except without the newer methods of getting codes/ids.